<div dir="rtl">

# تحويل الكتب العربية المصوّرة إلى Markdown

**طريقة الاستخدام (من الهاتف أو الحاسوب):**

1. من القائمة أعلاه: **Runtime ← Run all** (أو شغّل الخلايا بالترتيب بالضغط على ▶).
2. الخلية الأولى تثبّت الأدوات — تستغرق عدة دقائق أول مرة.
3. الخلية الثانية تطلب منك اختيار ملف PDF من جهازك، ثم تحوّله صفحة صفحة، وفي النهاية تنزّل ملف ZIP فيه `book.md` وكل الصفحات.

⚠ **ملاحظة خصوصية:** الكتاب يُعالَج على خوادم Google (Colab) لا على جهازك.

</div>

In [ ]:
#@title ١) التثبيت (مرة واحدة لكل جلسة — عدة دقائق)
%pip install -q "paddlepaddle==3.3.1" "paddleocr[doc-parser]==3.7.0" pymupdf
!rm -rf /content/book_ocr && git clone -q https://github.com/7aidaraa/book_ocr /content/book_ocr
import sys
sys.path.insert(0, "/content/book_ocr")
print("\u2713 التثبيت اكتمل — شغّل الخلية التالية")

In [ ]:
#@title ٢) رفع الكتاب وتحويله ثم تنزيل النتيجة
import os, shutil, sys
from pathlib import Path
from google.colab import files

sys.path.insert(0, "/content/book_ocr")
os.chdir("/content/book_ocr")

print("اختر ملف PDF من جهازك:")
uploaded = files.upload()
pdf_name = next(iter(uploaded))
pdf_path = Path("data/input") / pdf_name
pdf_path.parent.mkdir(parents=True, exist_ok=True)
Path(pdf_name).rename(pdf_path)

from app.book import process_book
from app.engines.paddleocr_engine import PaddleOCREngine

def on_progress(page, total, message):
    print(f"[{page}/{total}] {message}")

engine = PaddleOCREngine(lang="ar")
meta = process_book(pdf_path, engine, on_progress=on_progress)

book_dir = Path("data/output") / meta["book_name"]
zip_path = shutil.make_archive(f"/content/{meta['book_name']}", "zip", book_dir)

failed = meta["failed_pages"]
print(f"\n\u2713 تم: {meta['page_count']} صفحة" + (f"، فشل منها {len(failed)}: {failed}" if failed else " — كلها نجحت"))
print("يبدأ تنزيل ملف ZIP الآن...")
files.download(zip_path)